In [ ]:
import pandas as pd
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pulp
import os
from sklearn.preprocessing import MinMaxScaler


file1_path = '附件1.xlsx'
file2_path = '附件2.xlsx'

In [ ]:

file1_sheet1 = pd.read_excel(file1_path, sheet_name=0)
file1_sheet2 = pd.read_excel(file1_path, sheet_name=1)

file2_sheet1 = pd.read_excel(file2_path, sheet_name=0)
file2_sheet2 = pd.read_excel(file2_path, sheet_name=1)

In [ ]:
file2_sheet1=file2_sheet1.fillna(method='ffill')

In [ ]:
merged_df = pd.merge(file2_sheet1, file2_sheet2, on='作物编号', how='inner')
merged_df = pd.merge(file2_sheet1, file2_sheet2, on='作物编号', suffixes=('_x', '_y'))

columns_to_keep = [col for col in merged_df.columns if col.endswith('_x')]
columns_to_keep = [col.replace('_x', '') for col in columns_to_keep]

columns_to_drop = [col for col in merged_df.columns if col.endswith('_y')]

merged_df = merged_df.drop(columns=columns_to_drop)
merged_df.columns = [col.replace('_x', '') for col in merged_df.columns]

In [ ]:
if '亩产量/斤' in merged_df.columns and '销售单价/(元/斤)' in merged_df.columns:
    merged_df['销售量/斤'] = merged_df['亩产量/斤'] * merged_df['种植面积/亩']

In [ ]:
merged_df.to_excel('merged_file_with_sales.xlsx', index=False)

In [ ]:
sales_merged_df = merged_df.groupby('作物名称').agg({'销售量/斤': 'sum'}).reset_index()

In [ ]:
sales_merged_df .to_excel('各农作物总销售量.xlsx', index=False)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei'] 
plt.rcParams['axes.unicode_minus'] = False 
sales_merged_df = pd.read_excel('merged_file_with_sales.xlsx')

In [ ]:
import numpy as np
def calculate_final_price(row):
    price_range = row['销售单价/(元/斤)'].split('-')
    min_price = float(price_range[0])
    max_price = float(price_range[1])
    
    avg_price = (min_price + max_price) / 2
    
    random_fluctuation = np.random.uniform(-0.1, 0.1) * avg_price
    
    final_price = avg_price + random_fluctuation
    
    return final_price

df =  merged_df

df['最终销售价格/(元/斤)'] = df.apply(calculate_final_price, axis=1)

In [ ]:
df = df.groupby('作物名称').agg({'最终销售价格/(元/斤)': 'mean','销售量/斤': 'sum'}).reset_index()

In [ ]:
df2=pd.read_excel('初始数据未进行计算单价.xlsx')

In [ ]:
df2['加权销售价格/(元/斤)'] = df2.apply(calculate_final_price, axis=1)

In [ ]:

df2['总销量'] = df2['种植面积/亩'] * df2['亩产量/斤']
df2['总成本'] = df2['种植成本/(元/亩)'] * df2['种植面积/亩']
df2['总产量'] = df2['总销量']

In [ ]:
# 计算相关系数矩阵
corr = df2.select_dtypes(["float64", "int64"]).corr()

In [ ]:
Data= pd.read_excel('逻辑化数据.xlsx')

In [ ]:
# 应用函数逐行计算
Data['最终销售价格/(元/斤)'] = Data.apply(calculate_final_price, axis=1)

In [ ]:
Data

In [ ]:
import numpy as np
df=Data
df['作物类型'] = pd.factorize(df['作物类型'])[0]

ls1 = []  
ls2 = []  
ls3 = []  
ls4 = []  
ls5 = []  
ls6 = [] 
ls7 = [] 

def classify_row(row):
   
    if row['作物类型'] == 0 and row['作物名称'] != '水稻':
        ls1.append(row.name)
    elif row['作物名称'] == '水稻':
        ls2.append(row.name)
    elif (row['地块类型'] in [4, 5]) and row['作物类型'] != 2:
        ls6.append(row.name)
    elif row['作物类型'] == 2:
        ls5.append(row.name)
    elif row['种植季次'] == '第一季':
        ls3.append(row.name)
    elif row['种植季次'] == '第二季':
        ls4.append(row.name)
    if row['作物编号'] in [35, 36, 37]:
        ls7.append(row.name)

df.apply(classify_row, axis=1)

bean = df.index[df['是否豆类'] == '是'].tolist()


In [ ]:
df1=pd.read_excel('附件1sheet1.xlsx')

In [ ]:
def find_indices(lst, element):
    return [index for index, value in enumerate(lst) if value == element]
def function(sales_min1,sales_min2,sales_min3):
    for i in sales_min1.keys():
        total = 0
        index_ls = find_indices(land_types, i)

        total_product1 = sum([yields[j]*areas[i]*x[i][j][o]*0.25 for i in range(num_plots) for j in index_ls for o in range(2)])
        total_product2 = sum([yields[j]*areas[i]*x[i][j][o]*0.25 for i in range(num_plots) for j in index_ls for o in range(2,4)])
        total_product3 = sum([yields[j]*areas[i]*x[i][j][o]*0.25 for i in range(num_plots) for j in index_ls for o in range(4,6)])

        if total_product1 <= sales_min1[i]:
            pass
        else:
            total += (total_product-sales_min1[i])*prices[index_ls[0]]
        if total_product2 <= sales_min2[i]:
            pass
        else:
            total += (total_product-sales_min2[i])*prices[index_ls[0]]
            
        if total_product3 <= sales_min3[i]:
            pass
        else:
            total += (total_product-sales_min3[i])*prices[index_ls[0]]
    return total
def find_indices(lst, element):
    return [index for index, value in enumerate(lst) if value == element]

In [ ]:
import pulp
model = pulp.LpProblem("Land_Type_Optimization", pulp.LpMaximize)

In [ ]:
num_plots = 54

In [ ]:
num_crops = 125 

In [ ]:
land_types =  df['作物编号'].tolist() 

plot_type = df1['地块类型'].tolist() 
plot_area = df1['地块面积/亩'].tolist()
yields = df['亩产量/斤'].tolist()

In [ ]:

x = pulp.LpVariable.dicts(
    "planting_decision",               
    (range(num_plots), range(num_crops), range(6)),  
    cat="Binary"            
)

cost_per_acre = df['种植成本/(元/亩)'].tolist()

cos = [cost_per_acre] * num_plots 

In [ ]:
land_types =  df['作物编号'].tolist() 
plot_type = df1['地块类型'].tolist() 
plot_area = df1['地块面积/亩'].tolist() 

cost_per_acre = df['种植成本/(元/亩)'].tolist()


cos = [cost_per_acre] * num_plots 
sales = (df.groupby('作物编号').sum()['种植产量']*1).to_dict()
prices = (df['亩产量/斤']*df['最终销售价格/(元/斤)']).tolist() 
yields = df['亩产量/斤'].tolist() 
areas = df1['地块面积/亩'].tolist()  
costs=cos 

In [ ]:

def get_growth_rate(crop_id):
    if crop_id in [6, 7]:
      
        return np.random.normal(loc=0.075, scale=0.02)
    else:

        return np.random.normal(loc=0, scale=0.025)


salemax1_dict = {}
salemax2_dict = {}
salemax3_dict = {}


for crop_id, group in df.groupby('作物编号'):
    crop_type = group['作物类型'].iloc[0]  
    growth_rate = get_growth_rate(crop_id) 

    if crop_id in [6, 7]:
        growth_rate = max(0.05, min(0.10, growth_rate))
    else:
        growth_rate = max(-0.05, min(0.05, growth_rate))

    current_sales = group['种植产量'].sum()
    salemax1 = current_sales
    salemax2 = salemax1 * (1 + growth_rate)
    salemax3 = salemax2 * (1 + growth_rate)
    salemax1_dict[crop_id] = salemax1
    salemax2_dict[crop_id] = salemax2
    salemax3_dict[crop_id] = salemax3

In [ ]:

def get_growth_rate(crop_id):
    if crop_id in [6, 7]:
        return np.random.normal(loc=0.075, scale=0.02)  
    else:

        return np.random.normal(loc=0, scale=0.025) 

salemin1_dict = {}
salemin2_dict = {}
salemin3_dict = {}

for crop_id, group in df.groupby('作物编号'):
    crop_type = group['作物类型'].iloc[0]  
    growth_rate = get_growth_rate(crop_id)  

    if crop_id in [6, 7]:
        growth_rate = max(0.05, min(0.10, growth_rate))
    else:
        growth_rate = max(-0.05, min(0.05, growth_rate))


    current_sales = group['种植产量'].sum()
    salemin1 = current_sales
    salemin2 = salemin1 * (1 - abs(growth_rate))  
    salemin3 = salemin2 * (1 - abs(growth_rate))  

    salemin1_dict[crop_id] = salemin1
    salemin2_dict[crop_id] = salemin2
    salemin3_dict[crop_id] = salemin3

In [ ]:
import numpy as np


def get_yield_change():
   
    return np.random.normal(loc=1, scale=0.10)

yields1_list = []
yields2_list = []
yields3_list = []

for index, row in df.iterrows():
    initial_yield = row['亩产量/斤'] 
    
    yields1 = initial_yield
    yield_change1 = get_yield_change()
    yields2 = yields1 * yield_change1
    yield_change2 = get_yield_change()
    yields3 = yields2 * yield_change2

    yields1 = max(0, yields1)
    yields2 = max(0, yields2)
    yields3 = max(0, yields3)

    yields1_list.append(yields1)
    yields2_list.append(yields2)
    yields3_list.append(yields3)



In [ ]:
import numpy as np
import pandas as pd

def get_cost_increase():
    return np.random.normal(loc=1.05, scale=0.05) 
cost_per_acre = df['种植成本/(元/亩)'].tolist()


cos = [cost_per_acre] * num_plots
cost_df = pd.DataFrame()


for year in range(1, 4): 
    year_costs = np.zeros((num_plots, num_crops))
    
    for plot in range(num_plots):
        initial_costs = cos[plot] 
        
        for crop in range(num_crops):
            cost1 = initial_costs[crop]
            cost_increase1 = get_cost_increase()
            cost2 = cost1 * cost_increase1
            cost_increase2 = get_cost_increase()
            cost3 = cost2 * cost_increase2
            

            if year == 1:
                year_costs[plot, crop] = cost1
            elif year == 2:
                year_costs[plot, crop] = cost2
            elif year == 3:
                year_costs[plot, crop] = cost3


    year_df = pd.DataFrame(year_costs, columns=[f'作物_{i+1}' for i in range(num_crops)], index=[f'地块_{i+1}' for i in range(num_plots)])
    year_df.to_excel(f'年度成本_第{year}年.xlsx', engine='openpyxl')


    cost_df[f'第{year}年'] = year_df.stack()  


In [ ]:
import numpy as np

def get_price_change(crop_type, crop_id):
    if crop_type == 0:  
        return 1.0 
    elif crop_type == 1:  
        return np.random.normal(loc=1.05, scale=0.05) 
    elif crop_type == 2: 
        if crop_id == 89:  
            return np.random.normal(loc=0.95, scale=0.05)
        else: 
            return np.random.uniform(low=0.95, high=0.99) 
    else:
        return 1.0 

price1_list = []
price2_list = []
price3_list = []

for index, row in df.iterrows():

    initial_price = row['最终销售价格/(元/斤)']
    crop_type = row['作物类型']
    crop_id = row['作物编号'] 
    

    price1 = initial_price
    price_change1 = get_price_change(crop_type, crop_id)
    price2 = price1 * price_change1
    price_change2 = get_price_change(crop_type, crop_id)
    price3 = price2 * price_change2


    price1_list.append(price1)
    price2_list.append(price2)
    price3_list.append(price3)

In [ ]:
grain_crops = ls1  
rice_crop = ls2  
vegetable1_crops = ls3  
vegetable2_crops = ls4
fungi_crop = ls5  
vegetable_crops = ls6  

total_set = set(ls1 + ls2 + ls3 + ls4 + ls5 + ls6)


ex_grain_crops = list(total_set - set(grain_crops))
ex_rice_crop = list(total_set - set(rice_crop))
ex_vegetable1_crops = list(total_set - set(vegetable1_crops))
ex_vegetable2_crops = list(total_set - set(vegetable2_crops))
ex_fungi_crop = list(total_set - set(fungi_crop))
ex_vegetable_crops = list(total_set - set(vegetable_crops))
water_irrigated_second_season_crops = ls7
water_irrigated_first_season_crops = list(set(vegetable1_crops) - set(vegetable2_crops))

bean_crops = bean  


time = 6


In [ ]:
costs=[cost1_list,cost1_list,cost2_list,cost2_list,cost3_list,cost3_list]
prices=[price1_list,price1_list,price2_list,price2_list,price3_list,price3_list]
yields=[yields1_list,yields1_list,yields2_list,yields2_list,yields3_list,yields3_list]

In [ ]:

def get_growth_rate(crop_id):
    if crop_id in [6, 7]:
        
        return np.random.normal(loc=0.075, scale=0.02) 
    else:
  
        return np.random.normal(loc=0, scale=0.025)


def get_yield_change():
 
    return np.random.normal(loc=1, scale=0.10) 


def get_cost_increase():
  
    return np.random.normal(loc=1.05, scale=0.05) 


def get_price_change(crop_type, crop_id):
    if crop_type == 0: 
        return 1.0  
    elif crop_type == 1: 
        return np.random.normal(loc=1.05, scale=0.05) 
    elif crop_type == 2:  
        if crop_id == 89:
            return np.random.normal(loc=0.95, scale=0.05) 
        else:  
            return np.random.uniform(low=0.95, high=0.99)  
    else:
        return 1.0 


salemax1_dict = {}
salemax2_dict = {}
salemax3_dict = {}


salemin1_dict = {}
salemin2_dict = {}
salemin3_dict = {}


yields1_list = []
yields2_list = []
yields3_list = []


cost1_list = []
cost2_list = []
cost3_list = []


price1_list = []
price2_list = []
price3_list = []


for index, row in df.iterrows():
    crop_id = row['作物编号']
    crop_type = row['作物类型']

    growth_rate = get_growth_rate(crop_id)
    if crop_id in [6, 7]:
        growth_rate = max(0.05, min(0.10, growth_rate))
    else:
        growth_rate = max(-0.05, min(0.05, growth_rate))

    current_sales = row['种植产量']
    salemax1 = current_sales
    salemax2 = salemax1 * (1 + growth_rate)
    salemax3 = salemax2 * (1 + growth_rate)

    salemin1 = current_sales
    salemin2 = salemin1 * (1 - abs(growth_rate))
    salemin3 = salemin2 * (1 - abs(growth_rate))

    salemax1_dict[crop_id] = salemax1_dict.get(crop_id, 0) + salemax1
    salemax2_dict[crop_id] = salemax2_dict.get(crop_id, 0) + salemax2
    salemax3_dict[crop_id] = salemax3_dict.get(crop_id, 0) + salemax3

    salemin1_dict[crop_id] = salemin1_dict.get(crop_id, 0) + salemin1
    salemin2_dict[crop_id] = salemin2_dict.get(crop_id, 0) + salemin2
    salemin3_dict[crop_id] = salemin3_dict.get(crop_id, 0) + salemin3

    initial_yield = row['亩产量/斤']
    yields1 = initial_yield
    yield_change1 = get_yield_change()
    yields2 = yields1 * yield_change1
    yield_change2 = get_yield_change()
    yields3 = yields2 * yield_change2

    yields1_list.append(max(0, yields1))
    yields2_list.append(max(0, yields2))
    yields3_list.append(max(0, yields3))


    initial_cost = row['种植成本/(元/亩)']
    cost1 = initial_cost
    cost_increase1 = get_cost_increase()
    cost2 = cost1 * cost_increase1
    cost_increase2 = get_cost_increase()
    cost3 = cost2 * cost_increase2

    cost1_list.append(cost1)
    cost2_list.append(cost2)
    cost3_list.append(cost3)

    initial_price = row['最终销售价格/(元/斤)']
    price_change1 = get_price_change(crop_type, crop_id)
    price2 = initial_price * price_change1
    price_change2 = get_price_change(crop_type, crop_id)
    price3 = price2 * price_change2

    price1_list.append(initial_price)
    price2_list.append(price2)
    price3_list.append(price3)

In [ ]:
prices=price1_list
yields=yields1_list
cos = [cost_per_acre] * num_plots #种植成本
costs=cos 
prices= (yields1_list*df['最终销售价格/(元/斤)']).tolist() 
sales = (df.groupby('作物编号').sum()['种植产量']*1)

In [ ]:
import random

def group():
    global sales,sales_min1,sales_min2,sales_min3,yields,costs,price
    for i in sales.keys():
        if i == 6 or i == 7:
            sales[6] = sales[6]*(1+np.random.uniform(0.05,0.1))
            sales[7] = sales[7]*(1+np.random.uniform(0.05,0.1))
        else:
            sales[i] = sales[i]*(1+np.random.uniform(-0.05,0.05))
    sales_min1 = (sales*0.9).to_dict()

    for i in sales.keys():
        if i == 6 or i == 7:
            sales[6] = sales[6]*(1+np.random.uniform(0.05,0.1))
            sales[7] = sales[7]*(1+np.random.uniform(0.05,0.1))
        else:
            sales[i] = sales[i]*(1+np.random.uniform(-0.05,0.05))
    sales_min2 = (sales*0.9).to_dict()
    for i in sales.keys():
        if i == 6 or i == 7:
            sales[6] = sales[6]*(1+np.random.uniform(0.05,0.1))
            sales[7] = sales[7]*(1+np.random.uniform(0.05,0.1))
        else:
            sales[i] = sales[i]*(1+np.random.uniform(-0.05,0.05))
    sales_min3 = (sales*0.9).to_dict()

  
    yields = [i*(1+np.random.uniform(-0.1,0.1)) for i in yields]


    costs = [(np.array(costs[0])*1.05).tolist() for o in range(num_plots) ]

    for index,i in enumerate(df['作物类型'].values):
        if i == 1:
            prices[index] = prices[index]*1.05
        elif i == 2:
            prices[index] = prices[index]*(1+np.random.uniform(0.01,0.05))

In [ ]:
def opt():
    model = pulp.LpProblem("Land_Type_Optimization", pulp.LpMaximize)

    profit_sum = pulp.lpSum([
        (prices[j] * yields[j] - costs[i][j]) * areas[i] * x[i][j][t] 
        for i in range(num_plots)
        for j in range(num_crops)  
        for t in range(time)  
    ])

    penalty = function(sales_min1, sales_min2, sales_min3) 
   
    model += profit_sum - 0.5*penalty
    for i in range(num_plots):
        if plot_type[i] in [0, 1, 2]:
            for o in [0,2,4]:
             
                model += pulp.lpSum([x[i][j][o] for j in grain_crops]) ==3 
                model += pulp.lpSum([x[i][j][o+1] for j in range(num_crops)]) == 0 
            model += pulp.lpSum([x[i][j][o] for i in range(num_plots) for j in bean_crops for o in range(time)]) >= 1 
            for o in range(time):
                model += pulp.lpSum([x[i][j][o] for j in ex_grain_crops]) == 0
        elif plot_type[i] == 3:
            for o in [0,2,4]:
                model += pulp.lpSum([x[i][45][o]]) + pulp.lpSum([x[i][j][o] for j in vegetable1_crops]) == 1
                model += pulp.lpSum([x[i][j][o+1] for j in range(num_crops)]) == 0  
                model += pulp.lpSum([x[i][j][o] for j in vegetable1_crops]) == pulp.lpSum([x[i][j][o+1] for j in vegetable2_crops])
                model += pulp.lpSum([x[i][45][o+1]]) == 0
                model += pulp.lpSum([x[i][j][o+1] for j in ex_vegetable2_crops]) == 0
                model += pulp.lpSum([x[i][j][o] for j in ex_vegetable1_crops]) == 0
            model += pulp.lpSum([x[i][j][o] for i in range(num_plots) for j in bean_crops for o in range(time)]) >= 1
        elif plot_type[i] == 4:  # 普通大棚
            for o in [0,2,4]:
                model += pulp.lpSum([x[i][j][o] for j in vegetable_crops]) == 2 
                model += pulp.lpSum([x[i][j][o+1] for j in fungi_crop]) >=2 
                model += pulp.lpSum([x[i][j][o] for j in ex_vegetable_crops]) == 0 
                model += pulp.lpSum([x[i][j][o+1] for j in ex_fungi_crop]) == 0  
            model += pulp.lpSum([x[i][j][0]+x[i][j][2]+x[i][j][4] for j in bean_crops]) >= 1
        elif plot_type[i] == 5:  # 智慧大棚
            for o in [0,2,4]:
                model += pulp.lpSum([x[i][j][o] for j in vegetable1_crops]) == 3  
                model += pulp.lpSum([x[i][j][o+1] for j in vegetable1_crops]) == 3 
                model += pulp.lpSum([x[i][j][o] for j in ex_vegetable_crops]) == 0
            model += pulp.lpSum([x[i][j][o] for j in bean_crops for o in range(time)]) == 1 
  
    for j in range(num_crops):
        model += pulp.lpSum([x[i][j][o] for i in range(num_plots) for o in range(time)]) <= 2
        model += pulp.lpSum([x[i][j][o] for i in range(num_plots) if plot_type[i] == 4 for o in range(time)]) >= 2
    for i in range(num_plots):
         for j in range(num_crops):
                model += pulp.lpSum([x[i][j][0] + x[i][j][2]]) <= 1
                model += pulp.lpSum([x[i][j][2] + x[i][j][4]]) <= 1
                model += pulp.lpSum([x[i][j][1] + x[i][j][3]]) <= 1
                model += pulp.lpSum([x[i][j][3] + x[i][j][5]]) <= 1
    for j in range(num_crops):
        for o in range(6):
            model += pulp.lpSum([x[i][j][o] for i in range(num_plots)]) <= 3
    model.solve()
    output_folder = '问题2'
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    return x

df2=pd.read_excel('初始数据未进行计算单价.xlsx')
a = df2[['作物名称', '作物类型']].drop_duplicates()
dic = {i: o for i, o in zip(df2.作物名称, df2.作物类型)}
reward = pd.DataFrame(index=df['作物名称'].unique())
reward['作物类型'] = [dic[i] for i in reward.index]

start_year = 2024
end_year = 2030


with pd.ExcelWriter('问题2\\result.xlsx', engine='openpyxl', mode='w') as writer:
    for wh in range(3):
        group()
        x = opt()
        for t in range(6):
            current_year = 2023 + t//2 + 1 + wh*3
          
            if start_year <= current_year <= end_year:
                result = []
                product = []
                for i in range(num_plots):
                    tem = []
                    product_tem = []
                    for j in range(num_crops):
                        if pulp.value(x[i][j][t]) == 1:
                            tem.append(df1['地块面积/亩'].values[i])
                            product_tem.append(df1['地块面积/亩'].values[i] * df['亩产量/斤'].values[j])
                        else:
                            tem.append(0)
                            product_tem.append(0)

                    result.append(tem)
                    product.append(product_tem)

                result_df = pd.DataFrame(result, index=df1['地块名称'].values, columns=df['作物编号'].values).T
                result_df = result_df.groupby(result_df.index).sum().T
                result_df.columns = df['作物名称'].unique()

                for i, plot_name in enumerate(result_df.index):
                    total_area = result_df.loc[plot_name].sum() 
                    if total_area > plot_area[i]: 
                        scale_factor = plot_area[i] / total_area  
                        result_df.loc[plot_name] = result_df.loc[plot_name] * scale_factor  

            
                sheet_name = f'{current_year}年{t%2 + 1}季'
                result_df.to_excel(writer, sheet_name=sheet_name)

         
                product_df = pd.DataFrame(product, index=df1['地块名称'].values, columns=df['作物编号'].values).T
                product_df = product_df.groupby(product_df.index).sum().T
                product_df.columns = df['作物名称'].unique()
                reward[f'{current_year}年{t%2 + 1}季产量（斤）'] = product_df.sum()


with pd.ExcelWriter('问题2\\result.xlsx', engine='openpyxl', mode='a') as writer:
    reward.to_excel(writer, sheet_name='总产量')

In [ ]:
df['平均成本'] = df['亩产量/斤'] / df['种植成本/(元/亩)']

sale = df['种植产量'].values

cost = df.sort_values(by='作物编号').drop_duplicates(subset='作物名称')['平均成本'].values

price = df['最终销售价格/(元/斤)'].values

result = pd.DataFrame()
result.index = reward.index

for c in reward.columns[1:]:
    ls = []
    for i, o, n, m in zip(reward[c].values, sale, cost, price):
        if i > o:
            ls.append(o * (m - n) + (i - o) * (m - n) * 0.5)
        else:
            ls.append(i * (m - n))
    result[c + '盈利'] = ls


result['作物类型'] = reward['作物类型']

result_grouped = result.groupby('作物类型').sum()


output_path = '问题2/问题2地区所有季盈收情况汇总.xlsx'
result_grouped.to_excel(output_path)



In [ ]:
import plotly.express as px
import pandas as pd
result_grouped = pd.read_excel('问题2/情况二不同年份总的盈利情况.xlsx', index_col=0)


result_grouped = result_grouped.reset_index()  

fig = px.bar(
    result_grouped, 
    x='作物类型', 
    y=[col for col in result_grouped.columns if '盈利' in col], 
    title="不同作物类型的总盈利情况", 
    labels={'value': '总盈利（元）', 'variable': '年份'}, 
    barmode='group', 
)


fig.update_layout(
    xaxis_title='作物类型',
    yaxis_title='总盈利（元）',
    title_font_size=20,
    xaxis_tickangle=-45,  
    legend_title_text='年份',
    legend=dict(
        x=0.8, y=0.95, 
        bgcolor='rgba(255,255,255,0.5)' 
    )
)


fig.show()


In [ ]:
import plotly.express as px
import pandas as pd

df1 = pd.read_excel("问题1_1/问题1.1地区所有季盈收情况汇总.xlsx", index_col=0)
df2 = pd.read_excel("问题1_2/问题1.2地区所有季盈收情况汇总.xlsx", index_col=0)
df3 = pd.read_excel("问题2/问题2地区所有季盈收情况汇总.xlsx", index_col=0)


df1['来源'] = '问题1.1'
df2['来源'] = '问题1.2'
df3['来源'] = '问题2'


combined_df = pd.concat([df1, df2, df3])


combined_df = combined_df.reset_index()
combined_df = combined_df.melt(id_vars=['作物类型', '来源'], value_vars=[col for col in combined_df.columns if '盈利' in col], 
                               var_name='年份', value_name='总盈利（元）')

fig = px.bar(
    combined_df, 
    x='作物类型', 
    y='总盈利（元）', 
    color='来源',  
    facet_col='年份',  
    title="不同作物类型的总盈利情况对比", 
    labels={'总盈利（元）': '总盈利（元）', '来源': '数据来源'}, 
    barmode='group'
)


fig.update_layout(
    xaxis_title='作物类型',
    yaxis_title='总盈利（元）',
    title_font_size=20,
    xaxis_tickangle=-45,  
    legend_title_text='数据来源',
    legend=dict(
        x=0.8, y=0.95,  
        bgcolor='rgba(255,255,255,0.5)' 
    )
)


fig.show()


In [ ]:
import plotly.express as px
import pandas as pd

df1 = pd.read_excel("问题1_1/问题1.1地区所有季盈收情况汇总.xlsx", index_col=0)
df2 = pd.read_excel("问题1_2/问题1.2地区所有季盈收情况汇总.xlsx", index_col=0)
df3 = pd.read_excel("问题2/问题2地区所有季盈收情况汇总.xlsx", index_col=0)


df1['来源'] = '问题1.1'
df2['来源'] = '问题1.2'
df3['来源'] = '问题2'


combined_df = pd.concat([df1, df2, df3])


combined_df = combined_df.reset_index()
combined_df = combined_df.melt(id_vars=['作物类型', '来源'], value_vars=[col for col in combined_df.columns if '盈利' in col], 
                               var_name='年份', value_name='总盈利（元）')


fig = px.bar(
    combined_df, 
    x='作物类型', 
    y='总盈利（元）', 
    color='来源',  
    facet_col='年份',  
    title="不同作物类型的总盈利情况对比", 
    labels={'总盈利（元）': '总盈利（元）', '来源': '数据来源'}, 
    barmode='group'
)


fig.update_layout(
    width=1000, 
    height=600, 
    xaxis_title='作物类型',
    yaxis_title='总盈利（元）',
    title_font_size=24,
    xaxis_title_font_size=16,
    yaxis_title_font_size=16,
    xaxis_tickangle=-45,  
    legend_title_text='数据来源',
    legend=dict(
        x=1, y=1,  
        bgcolor='rgba(255,255,255,0.9)',  
        bordercolor='Black',
        borderwidth=1
    ),
    plot_bgcolor='rgba(240,240,240,0.8)', 
    paper_bgcolor='white'
)

fig.show()


In [ ]:
import plotly.express as px
import pandas as pd

df1 = pd.read_excel("问题1_1/问题1.1地区所有季盈收情况汇总.xlsx", index_col=0)
df2 = pd.read_excel("问题1_2/问题1.2地区所有季盈收情况汇总.xlsx", index_col=0)
df3 = pd.read_excel("问题2/问题2地区所有季盈收情况汇总.xlsx", index_col=0)

# 添加标识列来区分不同的文件
df1['来源'] = '问题1.1'
df2['来源'] = '问题1.2'
df3['来源'] = '问题2'


combined_df = pd.concat([df1, df2, df3])

combined_df = combined_df.reset_index()
combined_df = combined_df.melt(id_vars=['作物类型', '来源'], value_vars=[col for col in combined_df.columns if '盈利' in col], 
                               var_name='年份', value_name='总盈利（元）')

fig = px.bar(
    combined_df, 
    y='作物类型', 
    x='总盈利（元）', 
    color='来源', 
    facet_col='年份',
    facet_col_wrap=3,  
    title="不同作物类型的总盈利情况对比", 
    labels={'总盈利（元）': '总盈利（元）', '来源': '数据来源'}, 
    barmode='group',
    orientation='h' 
)


fig.update_layout(
    width=1500,  
    height=800, 
    xaxis_title='总盈利（元）',
    yaxis_title='作物类型',
    title_font_size=24,
    xaxis_title_font_size=16,
    yaxis_title_font_size=16,
    yaxis_tickangle=0,  
    legend_title_text='数据来源',
    legend=dict(
        x=1, y=1,  
        bgcolor='rgba(255,255,255,0.9)',  
        bordercolor='Black',
        borderwidth=1
    ),
    plot_bgcolor='rgba(240,240,240,0.8)',
    paper_bgcolor='white' 
)

fig.show()


In [ ]:
import plotly.express as px
import pandas as pd


df1 = pd.read_excel("问题1_1/问题1.1地区所有季盈收情况汇总.xlsx", index_col=0)
df2 = pd.read_excel("问题1_2/问题1.2地区所有季盈收情况汇总.xlsx", index_col=0)
df3 = pd.read_excel("问题2/问题2地区所有季盈收情况汇总.xlsx", index_col=0)


df1['来源'] = '问题1.1'
df2['来源'] = '问题1.2'
df3['来源'] = '问题2'

combined_df = pd.concat([df1, df2, df3])

combined_df = combined_df.reset_index()
combined_df = combined_df.melt(id_vars=['作物类型', '来源'], value_vars=[col for col in combined_df.columns if '盈利' in col], 
                               var_name='年份', value_name='总盈利（元）')

pivot_df = combined_df.pivot_table(index=['作物类型', '年份'], columns='来源', values='总盈利（元）').reset_index()
pivot_df['差距_问题1.1_vs_问题2'] = pivot_df['问题2'] - pivot_df['问题1.1']
pivot_df['差距_问题1.2_vs_问题2'] = pivot_df['问题2'] - pivot_df['问题1.2']

pivot_df_long = pivot_df.melt(id_vars=['作物类型', '年份'], value_vars=['差距_问题1.1_vs_问题2', '差距_问题1.2_vs_问题2'],
                               var_name='差距类型', value_name='差距（元）')

fig = px.line(
    pivot_df_long,
    x='年份',
    y='差距（元）',
    color='作物类型',
    line_dash='差距类型',
    markers=True,
    title="问题2中各作物与问题1.1和问题1.2的差距",
    labels={'差距（元）': '差距（元）', '差距类型': '差距对比'},
    facet_col='作物类型', 
    facet_col_wrap=3 
)

fig.update_layout(
    width=1500,
    height=800,
    xaxis_title='年份',
    yaxis_title='差距（元）',
    title_font_size=24,
    xaxis_title_font_size=16,
    yaxis_title_font_size=16,
    xaxis_tickangle=-45,
    legend_title_text='差距对比',
    legend=dict(
        x=1, y=1,
        bgcolor='rgba(255,255,255,0.9)',
        bordercolor='Black',
        borderwidth=1
    ),
    plot_bgcolor='rgba(240,240,240,0.8)',
    paper_bgcolor='white'
)

fig.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
plt.figure(figsize=(18, 10))

df1 = pd.read_excel("问题1_1/问题1.1地区所有季盈收情况汇总.xlsx", index_col=0)
df2 = pd.read_excel("问题1_2/问题1.2地区所有季盈收情况汇总.xlsx", index_col=0)
df3 = pd.read_excel("问题2/问题2地区所有季盈收情况汇总.xlsx", index_col=0)


df1['来源'] = '问题1.1'
df2['来源'] = '问题1.2'
df3['来源'] = '问题2'


combined_df = pd.concat([df1, df2, df3])


combined_df = combined_df.reset_index()
combined_df = combined_df.melt(id_vars=['作物类型', '来源'], value_vars=[col for col in combined_df.columns if '盈利' in col], 
                               var_name='年份', value_name='总盈利（元）')


pivot_df = combined_df.pivot_table(index=['作物类型', '年份'], columns='来源', values='总盈利（元）').reset_index()
pivot_df['差距_问题1.1_vs_问题2'] = pivot_df['问题2'] - pivot_df['问题1.1']
pivot_df['差距_问题1.2_vs_问题2'] = pivot_df['问题2'] - pivot_df['问题1.2']


pivot_df_long = pivot_df.melt(id_vars=['作物类型', '年份'], value_vars=['差距_问题1.1_vs_问题2', '差距_问题1.2_vs_问题2'],
                               var_name='差距类型', value_name='差距（元）')


crop_types = pivot_df_long['作物类型'].unique()


fig, axes = plt.subplots(nrows=len(crop_types), ncols=1, figsize=(10, 6 * len(crop_types)), sharex=True)


for ax, crop_type in zip(axes, crop_types):
    crop_df = pivot_df_long[pivot_df_long['作物类型'] == crop_type]
    
    for gap_type in crop_df['差距类型'].unique():
        gap_df = crop_df[crop_df['差距类型'] == gap_type]
        ax.plot(gap_df['年份'], gap_df['差距（元）'], marker='o', label=gap_type)

    ax.set_title(f'{crop_type} 差距分析')
    ax.set_xlabel('年份')
    ax.set_ylabel('差距（元）')
    ax.legend()


plt.tight_layout()
plt.show()
